# VN-GeoQA — End-to-End Notebook

Executable course notebook for ViGSQA: restore the frozen Vietnamese GeoQA benchmark, build the pinned OSM reference database, run the Direct and Text2SQL baselines through a llama.cpp server, and evaluate them.

Two run environments, identical otherwise:

- **Local** — reuse this repository checkout; the Pixi environment provides the Python runtime.
- **Google Colab** — section 0 clones the repository into `/content/ViGSQA` first.

Section 5 sets `RUN_MODE`: `smoke` (8 questions, 1 per type — integration evidence only, never benchmark evidence) or `full` (all 800 questions; official numbers belong to T03).


In [1]:
# ── 0. Bootstrap ──────────────────────────────────────────────────────────────
# Colab: clone the repository (HTTPS) and move into it. Local: reuse the checkout.
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    import subprocess
    if not os.path.isdir("/content/ViGSQA"):
        subprocess.run(
            ["git", "clone", "https://github.com/itskyf/ViGSQA.git", "/content/ViGSQA"],
            check=True,
        )
    %cd /content/ViGSQA
else:
    print(f"Local checkout: {os.getcwd()}")

Local checkout: /home/kpham/HCMUS/ViGSQA


## 1. Environment

Install the system packages (PostgreSQL/PostGIS, `osm2pgsql`, `osmium-tool`) and the Python dependencies. On Colab this uses `apt` and `pip`; locally the repository's Pixi environment already provides everything, so the script only verifies their presence.

In [2]:
# ── 1. Environment ────────────────────────────────────────────────────────────
!./scripts/install_dependencies.sh

if IN_COLAB:
    %pip install -q -r baselines/requirements.txt

[INFO] All required commands are available.
[INFO] Dependencies installation and service startup complete.


## 2. Frozen Benchmark (v1.0.0)

Restore the frozen VN-GeoQA benchmark from the public GitHub Release asset. The dataset is deliberately outside version control; `scripts/restore_dataset.sh` downloads, unpacks, and sha256-verifies it under `data/v1.0.0/questions_vi` (the tracked `generator/questions_vi` symlink points there). The script is idempotent — an existing verified copy is left untouched.


In [3]:
# ── 2. Dataset ────────────────────────────────────────────────────────────────
BENCHMARK_VERSION = "v1.0.0"

!./scripts/restore_dataset.sh

import glob, json

files = sorted(glob.glob(f"data/{BENCHMARK_VERSION}/questions_vi/*.jsonl"))
n = sum(1 for f in files for _ in open(f))
manifest = json.load(open(f"data/{BENCHMARK_VERSION}/questions_vi/MANIFEST.json"))
print(f"{len(files)} files, {n} questions, frozen={manifest['frozen']}, seed={manifest['seed']}")
assert n == 800


VN-GeoQA v1.0.0: 8 files, 800 questions verified.


8 files, 800 questions, frozen=True, seed=42


## 3. Reference Database (pinned OSM snapshot)

The benchmark was generated from the 2026-08-25 Geofabrik extract, so evaluation must run against that same snapshot: `OSM_URL` below pins the dated extract. The `vietnam-latest` default of `download_osm.sh` is only for demos and future benchmark versions — it never reproduces a frozen one. Import is idempotent: a completed import of the same file is skipped.

In [4]:
# ── 3. Reference DB ───────────────────────────────────────────────────────────
%env OSM_URL=https://download.geofabrik.de/asia/vietnam-260825.osm.pbf

!./scripts/init_database.sh
!./scripts/download_osm.sh
!./scripts/import_osm.sh

import psycopg

conn = psycopg.connect(host="127.0.0.1", dbname="osm_vn", user="postgres", password="postgres")
print("pois rows:", conn.execute("SELECT COUNT(*) FROM pois").fetchone()[0])
conn.close()

env: OSM_URL=https://download.geofabrik.de/asia/vietnam-260825.osm.pbf


[INFO] Checking database 'osm_vn'...
[INFO] Database 'osm_vn' already exists.
[INFO] Configuring PostGIS extension and database credentials...
[INFO] Database 'osm_vn' initialization complete.


[INFO] Resolving latest OSM extract URL: https://download.geofabrik.de/asia/vietnam-260825.osm.pbf...


[INFO] OSM file already exists: vietnam-260825.osm.pbf
[INFO] Active OSM source configured: vietnam-260825.osm.pbf


[INFO] Refreshing views and spatial index...


[INFO] Already imported: vietnam-260825.osm.pbf


pois rows: 38223


## 4. Model Server (llama.cpp)

The baselines talk to llama.cpp's OpenAI-compatible `/v1` endpoint through `langchain_openai.ChatOpenAI` — the same client locally and on Colab. Locally, `podman compose up -d llama-cpp` serves `ornith-ai/Ornith-1.5-9B-GGUF:Q4_K_M` (alias `ornith`) on port 8080. On Colab there is no Docker, so the cell installs the official `llama-cpp` package and starts `llama serve` with the same model. The first start downloads several GB of weights; the health check waits for readiness.

In [5]:
# ── 4. Model ─────────────────────────────────────────────────────────────────
import os, subprocess, time, urllib.request

os.environ.setdefault("LLAMACPP_URL", "http://localhost:8080")

if IN_COLAB:
    !pip install -q llama-cpp
    subprocess.Popen(
        ["llama", "serve", "-hf", "ornith-ai/Ornith-1.5-9B-GGUF:Q4_K_M",
         "--alias", "ornith", "--host", "127.0.0.1", "--port", "8080"],
        stdout=open("/tmp/llama-server.log", "w"), stderr=subprocess.STDOUT,
    )
else:
    print("Local: ensure the compose service is up — podman compose up -d llama-cpp")

url = os.environ["LLAMACPP_URL"]
for _ in range(600):  # up to ~50 min for the first weight download
    try:
        urllib.request.urlopen(f"{url}/health", timeout=2)
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError(f"llama.cpp not healthy at {url}")

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="ornith", base_url=f"{url}/v1", api_key="not-needed", temperature=0)
print("health ok — ping:", model.invoke("ping").content[:60])

Local: ensure the compose service is up — podman compose up -d llama-cpp


health ok — ping: Ping received! 👋 I'm here and ready to help. What can I do f


## 5. Run Configuration

`RUN_MODE` selects the question set. `smoke` runs the first question of each of the 8 types — a deterministic integration subset that exists to catch wiring bugs early; **its numbers are not benchmark evidence**. `full` runs all 800 questions and is the mode official results come from (T03). Model, prompts, and code paths are identical between the two modes.

In [6]:
# ── 5. Run config ─────────────────────────────────────────────────────────────
RUN_MODE = "smoke"   # "smoke" | "full"
MODEL = "llamacpp:ornith"

print(f"RUN_MODE={RUN_MODE}  MODEL={MODEL}  benchmark={BENCHMARK_VERSION}")

RUN_MODE=smoke  MODEL=llamacpp:ornith  benchmark=v1.0.0


## 6. Dataset Exploration

The benchmark's 8 question types, their answer-set sizes (range questions carry full distance-ordered gold sets — large tails are a documented characteristic, not a defect), and the two Vietnamese surfaces: full diacritics and stripped.

In [7]:
# ── 6. EDA ────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for f in files:
    qtype = f.rsplit("/", 1)[-1][: -len(".jsonl")]
    for line in open(f):
        q = json.loads(line)
        rows.append({
            "type": qtype,
            "n_answers": len(q.get("answers", [])),
            "question": q["question"],
            "stripped": q["question_surfaces"]["stripped"],
        })
eda = pd.DataFrame(rows)
print(eda.groupby("type")["n_answers"].agg(["count", "mean", "max"]).round(1))
eda.sample(5, random_state=0)[["type", "question", "stripped"]]

                      count  mean   max
type                                   
knn+distance            100   1.0     1
knn+loc                 100   1.0     1
knn+name                100   1.0     1
knn:direction+name      100   1.0     1
range+count             100   1.0     1
range+loc               100  28.7   533
range+name              100  47.2  1254
range:direction+name    100  34.0   868


,type,question,stripped
299,knn+name,Bạn có thể gợi ý tiệm bánh gần Cà Phê Nest By ...,Ban co the goi y tiem banh gan Ca Phe Nest By ...
500,range+loc,quán cà phê nào cách Tiểu học Tân Phú Thạnh 2 ...,quan ca phe nao cach Tieu hoc Tan Phu Thanh 2 ...
303,knn:direction+name,sân vận động gần nhất ở phía nam của thành kha...,san van dong gan nhat o phia nam cua thanh kha...
40,knn+distance,Từ Bia Hoi & Nuóc Mía đến khách sạn gần nhất l...,Tu Bia Hoi & Nuoc Mia den khach san gan nhat l...
495,range+count,Bao nhiêu khách sạn nằm trong vòng bán kính 5 ...,Bao nhieu khach san nam trong vong ban kinh 5 ...


## 7. Direct Baseline

The model answers each question directly, with no database in the loop. The runner is invoked as a subprocess so the notebook exercises exactly the CLI path (including its caches and evaluation) that a terminal run would.

In [8]:
# ── 7. Direct baseline ────────────────────────────────────────────────────────
!python baselines/baselines_vi.py --model {MODEL} --baseline direct --mode {RUN_MODE}

[nltk_data] Downloading package punkt to /home/kpham/nltk_data...


[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kpham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Model: llamacpp:ornith  |  Baseline: direct


Loaded 8 questions from /home/kpham/HCMUS/ViGSQA/generator/questions_vi
[nltk_data] Downloading package punkt to /home/kpham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kpham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!

[direct baseline]
  direct_json_parse: 100%|███████████████████| 8/8 [00:00<00:00, 568719.19it/s]


  Saved eval CSVs: llamacpp:ornith_direct_*.csv

Done.


## 8. Text2SQL Baseline

Question → SQL against the documented `pois` schema → read-only execution on the pinned database → answer rendered from the result records.

In [9]:
# ── 8. Text2SQL baseline ──────────────────────────────────────────────────────
!python baselines/baselines_vi.py --model {MODEL} --baseline text2sql --mode {RUN_MODE}

[nltk_data] Downloading package punkt to /home/kpham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kpham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Model: llamacpp:ornith  |  Baseline: text2sql


Loaded 8 questions from /home/kpham/HCMUS/ViGSQA/generator/questions_vi
[nltk_data] Downloading package punkt to /home/kpham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kpham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!

[text2sql baseline]
  sql_json_parse: 100%|██████████████████████| 8/8 [00:00<00:00, 578524.69it/s]


  Saved eval CSVs: llamacpp:ornith_text2sql_*.csv

Done.


## 9. Evaluation

Per-question metrics from the eval CSVs the runners wrote (`baselines/{model}_{baseline}_{text,parsed}_eval.csv`). In `smoke` mode these numbers are **integration-only**: they prove the wiring works, not model quality.

In [10]:
# ── 9. Evaluation ─────────────────────────────────────────────────────────────
frames = {}
for baseline in ("direct", "text2sql"):
    df = pd.read_csv(f"baselines/{MODEL}_{baseline}_text_eval.csv")
    df["baseline"] = baseline
    frames[baseline] = df

comp = pd.concat(frames.values(), ignore_index=True)
if RUN_MODE == "smoke":
    print("SMOKE — integration evidence only, not benchmark evidence.")
display(comp)

SMOKE — integration evidence only, not benchmark evidence.


,attempted,P,R,F1,type,id,baseline
0,True,0.000000,0.000000,0.000000,knn+distance,knn+distance-001,direct
1,False,0.000000,0.000000,0.000000,knn+loc,knn+loc-001,direct
2,True,0.000000,0.000000,0.000000,knn+name,knn+name-001,direct
3,True,0.000000,0.000000,0.000000,knn:direction+name,knn:direction+name-001,direct
4,True,0.000000,0.000000,0.000000,range+count,range+count-001,direct
5,False,0.000000,0.000000,0.000000,range+loc,range+loc-001,direct
6,True,0.027027,0.096774,0.042254,range+name,range+name-001,direct
7,True,0.008850,0.333333,0.017241,range:direction+name,range:direction+name-001,direct
8,True,1.000000,1.000000,1.000000,knn+distance,knn+distance-001,text2sql
9,False,0.000000,0.000000,0.000000,knn+loc,knn+loc-001,text2sql


## 10. Baseline Comparison

Direct vs Text2SQL, overall and by question type (joined back to the benchmark via the stable string ids).

In [11]:
# ── 10. Comparison ────────────────────────────────────────────────────────────
cols = [c for c in ("attempted", "F1") if c in comp.columns]
display(comp.groupby(["baseline", "type"])[cols].mean().round(3))


attempted     F1
baseline type                                  
direct   knn+distance                1.0  0.000
         knn+loc                     0.0  0.000
         knn+name                    1.0  0.000
         knn:direction+name          1.0  0.000
         range+count                 1.0  0.000
         range+loc                   0.0  0.000
         range+name                  1.0  0.042
         range:direction+name        1.0  0.017
text2sql knn+distance                1.0  1.000
         knn+loc                     0.0  0.000
         knn+name                    1.0  1.000
         knn:direction+name          1.0  0.600
         range+count                 1.0  1.000
         range+loc                   0.0  0.000
         range+name                  1.0  0.118
         range:direction+name        1.0  1.000

## 11. Error Analysis (pipeline level)

Where the runs fail *mechanically*: SQL execution errors, empty query results, unanswered types. The scientific error taxonomy over the full benchmark belongs to T03/T05; this section only proves the pipeline surfaces its own failures instead of hiding them.

In [12]:
# ── 11. Error analysis ────────────────────────────────────────────────────────
import json as _json

exec_path = f"baselines/cache_vi/{MODEL}/sql_exec.json"
try:
    cached = _json.load(open(exec_path))
    errors = [r for q in cached for r in q["records"] if r.get("error")]
    empty = [r for q in cached for r in q["records"] if not r.get("error") and not r.get("output")]
    print(f"sql_exec: {len(cached)} questions, {len(errors)} errored, {len(empty)} empty")
    for e in errors[:5]:
        print(" -", e["error"][:120])
except FileNotFoundError:
    print("no text2sql cache (direct-only run?)")

print("\nattempted rate by baseline:")
display(comp.groupby("baseline")["attempted"].mean().round(3))

sql_exec: 8 questions, 0 errored, 0 empty

attempted rate by baseline:


baseline
direct      0.75
text2sql    0.75
Name: attempted, dtype: float64

## 12. Demo — New Vietnamese Questions

Hand-written questions outside the benchmark, answered through the Text2SQL path against the same pinned database: generate SQL, execute it read-only, render the answer from the returned records (the exact prompt and record shapes the baseline runner uses).

In [13]:
# ── 12. Demo ──────────────────────────────────────────────────────────────────
import sys

sys.path.insert(0, "baselines")
import baselines as b
import baselines_vi  # noqa: F401 — applies the Vietnamese patches on import

gen_prompt = open(b.PROMPT_FILES["sql_generate"]).read()
answer_prompt = open(b.PROMPT_FILES["sql_answer"]).read()
demo_model = b.build_model(MODEL)
conn = b.make_db_conn()

DEMO_QUESTIONS = [
    "bệnh viện nào gần vị trí Bến Thành nhất?",
    "có bao nhiêu quán cà phê trong bán kính 2 km quanh Nhà thờ Đức Bà?",
]

for question in DEMO_QUESTIONS:
    print("❓", question)
    sql_text = demo_model.invoke([("system", gen_prompt), ("user", question)]).content
    blocks = b.extract_sql_blocks(sql_text)
    print("SQL:", blocks[0].strip() if blocks else sql_text[:200])
    if blocks:
        result = b.run_sql(blocks[0], conn)
        print("rows:", result["output"][:3], "error:", result["error"] or "none")
        if result["output"] and not result["error"]:
            records = json.dumps([{"output": result["output"], "error": ""}], indent=2)
            answer = demo_model.invoke([("system", answer_prompt + records), ("user", question)]).content
            print("A:", answer.strip()[:200])
    print("-" * 70)

[nltk_data] Downloading package punkt to /home/kpham/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kpham/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


❓ bệnh viện nào gần vị trí Bến Thành nhất?


SQL: SELECT p.poi_name
FROM pois p
CROSS JOIN (SELECT geometry FROM pois WHERE poi_name ILIKE '%Bến Thành%' LIMIT 1) ref
WHERE p.amenity ILIKE '%hospital%'
ORDER BY p.geometry <-> ref.geometry LIMIT 1
rows: [{'poi_name': 'Columbia Asia Saigon'}] error: none


A: Columbia Asia Saigon
----------------------------------------------------------------------
❓ có bao nhiêu quán cà phê trong bán kính 2 km quanh Nhà thờ Đức Bà?


SQL: SELECT COUNT(*) AS count
FROM pois p
WHERE ST_DWithin(p.geometry, (SELECT geometry FROM pois WHERE poi_name ILIKE '%Nhà thờ Đức Bà%' LIMIT 1), 2000)
  AND p.amenity ILIKE '%cafe%'
rows: [{'count': 246}] error: none


A: 246
----------------------------------------------------------------------


## 13. Extension Point (T04)

The project's methodological contribution — a typed deterministic answer renderer, selected (or rejected) on the strength of frozen-baseline error evidence — plugs into `baselines/baselines_vi.py`: the same patch layer this notebook already exercises, exposed as an additional `--baseline` option. No pipeline rewrite; this notebook would simply gain one more run cell like sections 7–8.